# AULA 03 — SQL NO GOOGLE COLAB
## NOTEBOOK DO ALUNO — prática guiada

Nesta aula, vamos trabalhar **SQL aplicado à análise de dados de e-commerce** usando três tabelas: `customers`, `orders` e `order_items`.

O notebook foi organizado para sair do básico e chegar a consultas mais próximas das tarefas de um analista de dados: filtros, agregações, `GROUP BY`, `ORDER BY`, `JOIN`, qualidade de dados, `CASE WHEN`, datas, `HAVING`, subconsultas, CTEs e funções de janela.

### Objetivos de aprendizagem
- entender tabelas, colunas, chaves e relacionamentos;
- consultar e filtrar dados com SQL;
- resumir informações com funções de agregação;
- relacionar tabelas com `JOIN`;
- investigar dados ausentes, inconsistentes e duplicados;
- responder perguntas de negócio usando SQL;
- interpretar resultados e comunicar conclusões.

> **Importante:** SQL é a linguagem. Neste Colab usaremos **SQLite** como banco de dados, pois ele funciona localmente no ambiente do notebook.

# 0. Correção do erro do `ipython-sql`

Em algumas versões atuais do Google Colab, o `ipython-sql` pode executar a consulta corretamente e falhar **somente ao tentar exibir o resultado**, gerando erros como:

```text
KeyError: 'DEFAULT'
```

ou

```text
KeyError: 'PLAIN_COLUMNS'
```

Neste notebook usamos a configuração que resolve esse problema no ambiente em que a aula foi testada:

```python
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%config SqlMagic.autopandas = True
```

**Não substitua `_DEPRECATED_DEFAULT` por `PLAIN_COLUMNS` neste material.**

`autopandas = True` também faz com que os resultados sejam entregues como DataFrames do pandas, o que é útil para análise de dados.

## 0.1 Instalar e configurar a extensão SQL

Execute esta célula **antes de qualquer célula `%%sql`**.

Se você já tiver carregado outra extensão SQL na mesma sessão e aparecer comportamento estranho, use **Ambiente de execução → Reiniciar sessão** e execute o notebook novamente desde o início.

In [ ]:
!pip -q install ipython-sql

%load_ext sql

# Correção da incompatibilidade de estilo de exibição no Colab
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# Faz os resultados SQL serem exibidos como DataFrames do pandas
%config SqlMagic.autopandas = True

print("✅ Extensão SQL carregada e configurada.")

## 0.2 Diagnóstico do ambiente

Esta célula é opcional, mas ajuda a conferir quais versões estão sendo usadas.

In [ ]:
from importlib.metadata import version, PackageNotFoundError

for pacote in ["ipython-sql", "prettytable", "pandas"]:
    try:
        print(f"{pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"{pacote}: não encontrado")

# 1. Upload dos arquivos

Envie os três arquivos usados na aula:

- `customers.csv`
- `orders.csv`
- `order_items.csv`

In [ ]:
from google.colab import files

uploaded = files.upload()

## 1.1 Conferir se todos os arquivos necessários foram enviados

Em análise de dados, vale a pena validar os arquivos **antes** de continuar o processo.

In [ ]:
import os

arquivos_necessarios = [
    "customers.csv",
    "orders.csv",
    "order_items.csv"
]

faltando = [arq for arq in arquivos_necessarios if not os.path.exists(arq)]

if faltando:
    raise FileNotFoundError(
        "Estão faltando estes arquivos: " + ", ".join(faltando)
    )

print("✅ Todos os arquivos foram encontrados.")

# 2. Ler os dados com pandas

Antes de criar o banco SQL, vamos carregar os arquivos como DataFrames para conferir quantidade de linhas, colunas e nomes dos campos.

In [ ]:
import pandas as pd
import sqlite3

customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)

## 2.1 Conhecer as colunas

In [ ]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nORDER_ITEMS")
print(order_items.columns.tolist())

## 2.2 Visualização inicial

`head()` mostra algumas linhas e ajuda a entender rapidamente como os dados estão organizados.

In [ ]:
display(customers.head())
display(orders.head())
display(order_items.head())

# 3. Criar o banco SQLite

Os DataFrames serão gravados em um arquivo chamado `aula03_ecommerce.db`.

`if_exists="replace"` significa que, se a tabela já existir, ela será substituída. Isso é conveniente para a aula porque permite executar novamente a célula sem criar tabelas duplicadas.

In [ ]:
con = sqlite3.connect("aula03_ecommerce.db")

customers.to_sql("customers", con, if_exists="replace", index=False)
orders.to_sql("orders", con, if_exists="replace", index=False)
order_items.to_sql("order_items", con, if_exists="replace", index=False)

con.close()

print("✅ Banco aula03_ecommerce.db criado com sucesso!")

# 4. Conectar o `ipython-sql` ao banco SQLite

In [ ]:
%sql sqlite:///aula03_ecommerce.db

## 4.1 Teste de funcionamento

Se esta consulta mostrar as três tabelas, o ambiente está funcionando.

In [ ]:
%%sql
SELECT name AS tabela
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

# 5. Modelo de dados

Neste conjunto:

**`customers`**
- contém informações dos clientes;
- usamos `customer_id` para relacioná-la a `orders`;
- contém cidade e estado.

**`orders`**
- contém os pedidos;
- possui status, datas de compra e entrega;
- relaciona clientes e itens.

**`order_items`**
- contém os itens de cada pedido;
- possui preço e frete.

### Relações principais

```text
customers.customer_id
        ↓
orders.customer_id

orders.order_id
        ↓
order_items.order_id
```

## 5.1 Inspecionar a estrutura das tabelas

In [ ]:
%%sql
PRAGMA table_info(customers);

In [ ]:
%%sql
PRAGMA table_info(orders);

In [ ]:
%%sql
PRAGMA table_info(order_items);

# 6. SELECT, FROM e LIMIT

A estrutura mais básica de uma consulta é:

```sql
SELECT colunas
FROM tabela;
```

`LIMIT` restringe a quantidade de linhas exibidas. É muito útil para explorar tabelas grandes sem trazer todos os registros.

In [ ]:
%%sql
SELECT *
FROM orders
LIMIT 10;

## 6.1 Selecionando apenas algumas colunas

Evitar `SELECT *` quando você não precisa de todas as colunas deixa a consulta mais clara e, em bancos maiores, pode melhorar desempenho.

In [ ]:
%%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp
FROM orders
LIMIT 10;

## Exercício 1 — seleção de colunas

Mostre apenas `customer_id`, `customer_city` e `customer_state` da tabela `customers`. Exiba somente 15 registros.

In [ ]:
%%sql
-- Use SELECT, FROM e LIMIT 15.

-- ESCREVA SUA CONSULTA AQUI

# 7. DISTINCT — valores únicos

`DISTINCT` remove repetições do resultado.

É útil para responder perguntas como: **quais status de pedido existem na base?**

In [ ]:
%%sql
SELECT DISTINCT order_status
FROM orders
ORDER BY order_status;

## Exercício 2 — estados presentes na base

Liste os estados distintos da tabela `customers` em ordem alfabética.

In [ ]:
%%sql
-- Use DISTINCT e ORDER BY.

-- ESCREVA SUA CONSULTA AQUI

# 8. WHERE — filtrando linhas

`WHERE` determina quais registros entram no resultado.

Exemplos de operadores:

| Operador | Significado |
|---|---|
| `=` | igual |
| `<>` ou `!=` | diferente |
| `>` | maior que |
| `<` | menor que |
| `>=` | maior ou igual |
| `<=` | menor ou igual |
| `IN (...)` | está em uma lista |
| `BETWEEN ... AND ...` | está em um intervalo |
| `LIKE` | corresponde a um padrão |
| `IS NULL` | valor ausente |

In [ ]:
%%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp
FROM orders
WHERE order_status = 'delivered'
LIMIT 10;

## 8.1 AND e OR

`AND` exige que **todas** as condições sejam verdadeiras.  
`OR` exige que **pelo menos uma** seja verdadeira.

In [ ]:
%%sql
SELECT
    customer_id,
    customer_city,
    customer_state
FROM customers
WHERE customer_state = 'RJ'
  AND customer_city = 'rio de janeiro'
LIMIT 10;

## Exercício 3 — múltiplos estados

Mostre clientes que estejam em `RJ`, `SP` ou `MG`. Use `IN`.

In [ ]:
%%sql
-- Dica: WHERE customer_state IN ('RJ', 'SP', 'MG')

-- ESCREVA SUA CONSULTA AQUI

# 9. ORDER BY — ordenação

`ORDER BY` **não agrupa** dados. Ele apenas define a ordem em que o resultado será mostrado.

- `ASC` → crescente;
- `DESC` → decrescente.

In [ ]:
%%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp
FROM orders
ORDER BY order_purchase_timestamp DESC
LIMIT 10;

# 10. Funções de agregação

Funções de agregação resumem várias linhas em um valor.

| Função | Uso |
|---|---|
| `COUNT()` | contar |
| `SUM()` | somar |
| `AVG()` | média |
| `MIN()` | menor valor |
| `MAX()` | maior valor |

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_pedidos
FROM orders;

In [ ]:
%%sql
SELECT
    ROUND(AVG(price), 2) AS preco_medio_item,
    ROUND(MIN(price), 2) AS menor_preco,
    ROUND(MAX(price), 2) AS maior_preco
FROM order_items;

## Exercício 4 — frete

Calcule o frete médio, o menor frete e o maior frete da tabela `order_items`.
Arredonde as médias para duas casas decimais.

In [ ]:
%%sql
-- Use AVG, MIN, MAX e ROUND.

-- ESCREVA SUA CONSULTA AQUI

# 11. GROUP BY — agrupando categorias

`GROUP BY` reúne registros com o mesmo valor para que possamos calcular métricas **por grupo**.

Exemplo: em vez de contar todos os pedidos, podemos contar pedidos **por status**.

In [ ]:
%%sql
SELECT
    order_status,
    COUNT(*) AS quantidade_pedidos
FROM orders
GROUP BY order_status
ORDER BY quantidade_pedidos DESC;

### Leitura da consulta

1. `FROM orders` → usamos a tabela de pedidos;
2. `GROUP BY order_status` → formamos um grupo para cada status;
3. `COUNT(*)` → contamos quantas linhas existem em cada grupo;
4. `ORDER BY ... DESC` → mostramos do maior para o menor.

**Resumo:** `GROUP BY` agrupa; `ORDER BY` ordena.

## Exercício 5 — clientes por estado

Conte quantos clientes existem em cada estado e organize do estado com mais clientes para o estado com menos clientes.

In [ ]:
%%sql
-- Use COUNT(*), GROUP BY e ORDER BY ... DESC.

-- ESCREVA SUA CONSULTA AQUI

# 12. HAVING — filtrando grupos

`WHERE` filtra **linhas antes do agrupamento**.  
`HAVING` filtra **grupos depois do `GROUP BY`**.

Exemplo: mostrar apenas estados com pelo menos 100 clientes.

In [ ]:
%%sql
SELECT
    customer_state,
    COUNT(*) AS quantidade_clientes
FROM customers
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY quantidade_clientes DESC;

## Exercício 6 — status com muitos pedidos

Conte os pedidos por status e mostre somente os status que tenham mais de 100 pedidos.

In [ ]:
%%sql
-- Use GROUP BY, HAVING COUNT(*) > 100 e ORDER BY.

-- ESCREVA SUA CONSULTA AQUI

# 13. JOIN — relacionando tabelas

Muitas perguntas de negócio exigem dados que estão em tabelas diferentes.

Para descobrir **quantos pedidos existem por estado**, precisamos de:

- `orders`: possui os pedidos e `customer_id`;
- `customers`: possui o estado e `customer_id`.

O `JOIN` combina as tabelas pela coluna em comum.

In [ ]:
%%sql
SELECT
    c.customer_state,
    COUNT(o.order_id) AS quantidade_pedidos
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY quantidade_pedidos DESC
LIMIT 10;

### O que são `o` e `c`?

São **aliases** (apelidos):

```sql
orders AS o
customers AS c
```

Assim podemos escrever `o.order_id` e `c.customer_state`, deixando claro de qual tabela vem cada coluna.

## Exercício 7 — pedidos entregues por estado

Mostre a quantidade de pedidos com status `delivered` por estado.

In [ ]:
%%sql
-- Use JOIN + WHERE + GROUP BY + ORDER BY.

-- ESCREVA SUA CONSULTA AQUI

# 14. JOIN com três tabelas

Para combinar estado do cliente, pedido e valores dos itens, usamos:

```text
customers → orders → order_items
```

A consulta abaixo calcula o frete médio dos itens por estado.

In [ ]:
%%sql
SELECT
    c.customer_state,
    ROUND(AVG(oi.freight_value), 2) AS frete_medio
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY frete_medio DESC;

## Exercício 8 — preço médio dos itens por estado

Calcule o preço médio dos itens (`price`) por estado do cliente. Mostre do maior preço médio para o menor.

In [ ]:
%%sql
-- Relacione customers → orders → order_items e use AVG(oi.price).

-- ESCREVA SUA CONSULTA AQUI

# 15. Qualidade de dados

Antes de confiar em uma métrica, devemos investigar a qualidade da base.

Um dado aparentemente estranho pode ser:
- ausente;
- duplicado;
- inconsistente;
- impossível;
- ou simplesmente um valor raro, mas verdadeiro.

**Boa prática:** detectar → investigar → decidir → tratar → documentar.

## 15.1 Procurar valores ausentes

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS entrega_ausente,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS previsao_ausente
FROM orders;

## 15.2 Procurar estados ausentes

In [ ]:
%%sql
SELECT COUNT(*) AS clientes_sem_estado
FROM customers
WHERE customer_state IS NULL
   OR TRIM(customer_state) = '';

## 15.3 Procurar chaves duplicadas

In [ ]:
%%sql
SELECT
    order_id,
    COUNT(*) AS ocorrencias
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC;

## 15.4 Procurar valores financeiros potencialmente inválidos

In [ ]:
%%sql
SELECT *
FROM order_items
WHERE price < 0
   OR freight_value < 0
LIMIT 20;

## Exercício 9 — investigar possíveis inconsistências

Liste todos os valores distintos de `customer_state` em ordem alfabética.  
Depois observe se há valores ausentes ou grafias inesperadas.

In [ ]:
%%sql
-- Use DISTINCT e ORDER BY.

-- ESCREVA SUA CONSULTA AQUI

# 16. CASE WHEN — criando regras de classificação

`CASE WHEN` funciona como uma estrutura condicional dentro do SQL.

Vamos classificar pedidos entregues como **Atrasado** ou **No prazo**.

In [ ]:
%%sql
SELECT
    order_id,
    order_estimated_delivery_date,
    order_delivered_customer_date,
    CASE
        WHEN datetime(order_delivered_customer_date) > datetime(order_estimated_delivery_date)
            THEN 'Atrasado'
        ELSE 'No prazo'
    END AS situacao_entrega
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
LIMIT 20;

## Exercício 10 — classificar preço do item

Crie uma coluna `faixa_preco` com as regras:

- preço menor que 50 → `Baixo`;
- preço de 50 até 199,99 → `Médio`;
- preço igual ou maior que 200 → `Alto`.

Mostre `order_id`, `price` e `faixa_preco`.

In [ ]:
%%sql
-- Use CASE WHEN ... THEN ... WHEN ... THEN ... ELSE ... END.

-- ESCREVA SUA CONSULTA AQUI

# 17. Datas no SQLite

No SQLite, `date()` e `datetime()` ajudam a interpretar datas, enquanto `julianday()` permite calcular diferenças em dias.

In [ ]:
%%sql
SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    ROUND(
        julianday(order_delivered_customer_date)
        - julianday(order_purchase_timestamp),
        2
    ) AS dias_ate_entrega
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
LIMIT 20;

## 17.1 Tempo médio de entrega por estado

In [ ]:
%%sql
SELECT
    c.customer_state,
    ROUND(
        AVG(
            julianday(o.order_delivered_customer_date)
            - julianday(o.order_purchase_timestamp)
        ),
        2
    ) AS media_dias_entrega
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY media_dias_entrega DESC;

## Exercício 11 — pedidos por mês

Conte quantos pedidos foram realizados em cada mês.  
Dica: `strftime('%Y-%m', order_purchase_timestamp)`.

In [ ]:
%%sql
-- Use strftime('%Y-%m', order_purchase_timestamp), COUNT e GROUP BY.

-- ESCREVA SUA CONSULTA AQUI

# 18. Percentuais com SQL

Para calcular percentuais, normalmente dividimos uma contagem específica pela contagem total.

O `100.0` força o cálculo decimal.

In [ ]:
%%sql
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentual_entregues
FROM orders;

## Desafio 1 — percentual de atrasos por estado

Considere atraso quando:

```text
order_delivered_customer_date > order_estimated_delivery_date
```

Calcule:
- total de pedidos entregues com datas válidas;
- quantidade de pedidos atrasados;
- percentual de atrasos;
- tudo agrupado por estado.

In [ ]:
%%sql
-- Use CASE WHEN dentro de SUM(), COUNT(*), GROUP BY e ORDER BY.

-- ESCREVA SUA CONSULTA AQUI

# 19. Subconsultas

Uma subconsulta é uma consulta dentro de outra consulta.

Exemplo: descobrir itens com preço **acima da média geral**.

In [ ]:
%%sql
SELECT
    order_id,
    product_id,
    price
FROM order_items
WHERE price > (
    SELECT AVG(price)
    FROM order_items
)
ORDER BY price DESC
LIMIT 20;

## Exercício 12 — fretes acima da média

Mostre itens cujo `freight_value` esteja acima do frete médio de toda a tabela.

In [ ]:
%%sql
-- Use WHERE freight_value > (SELECT AVG(freight_value) ...).

-- ESCREVA SUA CONSULTA AQUI

# 20. CTE — Common Table Expression

Uma CTE começa com `WITH` e permite dividir consultas complexas em etapas mais legíveis.

A consulta abaixo calcula o valor dos itens de cada pedido e depois resume esses pedidos.

In [ ]:
%%sql
WITH valor_por_pedido AS (
    SELECT
        order_id,
        SUM(price) AS valor_itens,
        SUM(freight_value) AS valor_frete
    FROM order_items
    GROUP BY order_id
)
SELECT
    ROUND(AVG(valor_itens), 2) AS ticket_medio_itens,
    ROUND(AVG(valor_frete), 2) AS frete_medio_por_pedido
FROM valor_por_pedido;

## Exercício 13 — faturamento de itens por estado

Crie uma CTE que some `price` por `order_id`. Depois relacione-a a `orders` e `customers` para calcular o total de valor de itens por estado.

In [ ]:
%%sql
-- Comece com WITH valor_por_pedido AS (...) e depois faça os JOINs.

-- ESCREVA SUA CONSULTA AQUI

# 21. Funções de janela — introdução

Funções de janela calculam informações sobre um conjunto de linhas **sem transformar tudo em uma única linha por grupo**.

Exemplo: criar um ranking dos estados por quantidade de pedidos.

In [ ]:
%%sql
WITH pedidos_estado AS (
    SELECT
        c.customer_state,
        COUNT(o.order_id) AS quantidade_pedidos
    FROM orders AS o
    INNER JOIN customers AS c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
)
SELECT
    customer_state,
    quantidade_pedidos,
    RANK() OVER (ORDER BY quantidade_pedidos DESC) AS ranking
FROM pedidos_estado
ORDER BY ranking;

## Exercício 14 — ranking de frete médio

Monte um ranking dos estados do maior para o menor frete médio.
Dica: primeiro calcule o frete médio por estado em uma CTE; depois aplique `RANK() OVER (...)`.

In [ ]:
%%sql
-- Use CTE + AVG + RANK() OVER (ORDER BY ... DESC).

-- ESCREVA SUA CONSULTA AQUI

# 22. Perguntas de negócio — prática integrada

Agora o objetivo não é apenas escrever comandos, mas transformar uma pergunta em uma consulta e interpretar o resultado.

## Desafio 2 — taxa de cancelamento por estado

Calcule, por estado:
- quantidade total de pedidos;
- quantidade de pedidos cancelados;
- percentual de cancelamentos.

Ordene do maior percentual para o menor.

In [ ]:
%%sql
-- Use JOIN, CASE WHEN, SUM, COUNT e GROUP BY.

-- ESCREVA SUA CONSULTA AQUI

## Desafio 3 — valor médio dos pedidos por estado

Um pedido pode ter vários itens. Portanto:

1. some `price` por pedido;
2. depois calcule a média desses totais por estado.

**Atenção:** calcular `AVG(price)` diretamente responde ao preço médio dos **itens**, não ao valor médio dos **pedidos**.

In [ ]:
%%sql
-- Use uma CTE para somar price por order_id; depois AVG por estado.

-- ESCREVA SUA CONSULTA AQUI

# 23. Lista de exercícios de fixação

Resolva sem copiar consultas anteriores sempre que possível.

**Nível 1 — fundamentos**
1. Mostre 20 pedidos e seus status.
2. Liste os status existentes sem repetição.
3. Conte quantos registros existem em `order_items`.
4. Mostre os 10 itens de maior preço.
5. Liste clientes do estado de São Paulo.

**Nível 2 — agregação e relacionamento**
6. Conte pedidos por status.
7. Conte clientes por estado.
8. Calcule preço médio e frete médio dos itens.
9. Calcule quantidade de pedidos por estado.
10. Calcule o frete médio por estado.

**Nível 3 — análise**
11. Calcule a duração média de entrega por estado.
12. Calcule o percentual de atrasos por estado.
13. Calcule o percentual de cancelamentos por estado.
14. Calcule o valor médio dos pedidos por estado.
15. Crie um ranking dos estados por quantidade de pedidos.

## Espaço para resolver os exercícios de fixação

Crie novas células `%%sql` conforme necessário.

In [ ]:
%%sql
-- Exercício 1

-- ESCREVA SUA CONSULTA AQUI

In [ ]:
%%sql
-- Exercício 2

-- ESCREVA SUA CONSULTA AQUI

In [ ]:
%%sql
-- Exercício 3

-- ESCREVA SUA CONSULTA AQUI

In [ ]:
%%sql
-- Exercício 4

-- ESCREVA SUA CONSULTA AQUI

In [ ]:
%%sql
-- Exercício 5

-- ESCREVA SUA CONSULTA AQUI

In [ ]:
%%sql
-- Continue criando células para os exercícios 6 a 15.

-- ESCREVA SUA CONSULTA AQUI

# 25. Mini-case final — análise de e-commerce

Escolha **uma pergunta de negócio** que possa ser respondida com estas tabelas.

Sugestões:
- quais estados possuem maior volume de pedidos?
- onde o frete é proporcionalmente mais alto?
- quais estados apresentam maior atraso?
- como os pedidos evoluem ao longo dos meses?
- onde a taxa de cancelamento é maior?

Sua entrega deve conter:

**1. Pergunta de negócio**  
O que você quer descobrir?

**2. Consulta SQL**  
Qual código responde à pergunta?

**3. Evidência**  
Quais números ou padrões foram encontrados?

**4. Interpretação**  
O que esses números significam para o negócio?

**5. Limitações**  
O que os dados não permitem concluir?

**6. Próximo passo**  
Qual informação adicional você buscaria para aprofundar a análise?

In [ ]:
%%sql
-- Escreva aqui a consulta SQL do seu mini-case.

-- ESCREVA SUA CONSULTA AQUI

# 26. Checklist de raciocínio do analista

Antes de considerar uma análise pronta, confirme:

- A pergunta de negócio está clara?
- As tabelas usadas são as corretas?
- O `JOIN` usa a chave certa?
- Há `NULL` que podem distorcer a métrica?
- Existe risco de duplicar linhas após o `JOIN`?
- A métrica é de item, pedido ou cliente?
- A média escolhida responde à pergunta?
- O denominador do percentual está correto?
- O resultado precisa ser ordenado?
- A conclusão está realmente sustentada pelos dados?

# 27. Resumo dos principais comandos

| Comando | Função |
|---|---|
| `SELECT` | escolher colunas |
| `FROM` | indicar a tabela |
| `DISTINCT` | eliminar repetições do resultado |
| `WHERE` | filtrar linhas |
| `IN` | filtrar por uma lista de valores |
| `IS NULL` | localizar valores ausentes |
| `ORDER BY` | ordenar resultados |
| `ASC` / `DESC` | crescente / decrescente |
| `COUNT()` | contar |
| `SUM()` | somar |
| `AVG()` | calcular média |
| `MIN()` / `MAX()` | menor / maior valor |
| `GROUP BY` | agrupar linhas |
| `HAVING` | filtrar grupos |
| `JOIN` | relacionar tabelas |
| `CASE WHEN` | criar regras condicionais |
| `ROUND()` | arredondar |
| `datetime()` | interpretar data/hora |
| `julianday()` | apoiar cálculos entre datas |
| `strftime()` | extrair/formartar partes de datas |
| `WITH` | criar CTEs |
| `RANK() OVER()` | criar ranking sem perder linhas agregadas |

# 28. Se aparecer erro no `%%sql`

Confira, nesta ordem:

1. a célula de instalação do `ipython-sql` foi executada;
2. `%load_ext sql` foi executado;
3. `%config SqlMagic.style = '_DEPRECATED_DEFAULT'` foi executado;
4. `%config SqlMagic.autopandas = True` foi executado;
5. o banco foi criado;
6. `%sql sqlite:///aula03_ecommerce.db` foi executado;
7. a célula SQL começa com `%%sql` na primeira linha.

Se ainda houver conflito com uma extensão carregada anteriormente, reinicie a sessão e execute novamente **de cima para baixo**.